In [1]:
import numpy as np
import networkx as nx
import math
import time

def generate_dataset(n=100, m=80, lambda_=3, mu_=3):
    """Generates the similarity matrix S with a critical paper."""
    np.random.seed(42)
    # Generate random similarities in [0.2, 0.8]
    S = np.random.uniform(0.2, 0.8, (n, m))

    # Create "Critical Paper" at index 0
    # It has very low similarity with most reviewers
    S[:, 0] = np.random.uniform(0.0, 0.2, n)
    # But has 2 high-expertise reviewers
    S[0, 0] = 0.95
    S[1, 0] = 0.92

    return S, n, m, lambda_, mu_

def peer_review_4_all_simplified(S, lambda_, mu_):
    """
    Implements a simplified iterative Max-Flow procedure for Max-Min fairness
    OPTIMIZED with Binary Search for larger datasets.
    """
    n, m = S.shape
    assignment = {j: [] for j in range(m)}
    reviewer_caps = {i: mu_ for i in range(n)}
    papers_needed = {j: lambda_ for j in range(m)}

    # Flatten and sort similarities descending
    sims = []
    for i in range(n):
        for j in range(m):
            sims.append((S[i, j], i, j))
    sims.sort(key=lambda x: x[0], reverse=True)

    # Helper subroutine: Thresholded Max Flow using BINARY SEARCH
    def check_flow(k_target, active_papers):
        target_flow = sum(min(papers_needed[p], k_target) for p in active_papers)

        left = 0
        right = len(sims) - 1
        best_flow = None
        best_sim = -1

        while left <= right:
            mid = (left + right) // 2

            # Build graph with all edges down to the 'mid' threshold
            G = nx.DiGraph()
            G.add_node('Source')
            G.add_node('Sink')

            for i in range(n):
                if reviewer_caps[i] > 0:
                    G.add_edge('Source', f'R_{i}', capacity=reviewer_caps[i])

            for j in active_papers:
                needed = min(papers_needed[j], k_target)
                G.add_edge(f'P_{j}', 'Sink', capacity=needed)

            # Add all valid edges from index 0 up to 'mid'
            for idx in range(mid + 1):
                sim, i, j = sims[idx]
                if j in active_papers and reviewer_caps[i] > 0 and i not in assignment[j]:
                    G.add_edge(f'R_{i}', f'P_{j}', capacity=1)

            # Check flow
            flow_val, flow_dict = nx.maximum_flow(G, 'Source', 'Sink')

            if flow_val == target_flow:
                # Target met! Record this success.
                best_flow = flow_dict
                best_sim = sims[mid][0]
                # Now try to restrict edges further (move left to find a HIGHER minimum similarity)
                right = mid - 1
            else:
                # Target NOT met! We need MORE edges (move right to allow LOWER similarities)
                left = mid + 1

        return best_flow, best_sim

    # Outer Loop: Fix the worst-off paper iteratively
    active_papers = list(range(m))
    while active_papers:
        best_flow = None
        best_k = 1

        # Test capacities k in [1..lambda]
        for k in range(1, lambda_ + 1):
            flow_dict, min_sim = check_flow(k, active_papers)
            if flow_dict is not None:
                best_flow = flow_dict
                best_k = k

        if not best_flow:
            raise ValueError("No feasible flow found. Constraints too tight.")

        # Extract temporary assignments
        temp_scores = {p: 0 for p in active_papers}
        temp_assignment = {p: [] for p in active_papers}
        for i in range(n):
            if f'R_{i}' in best_flow:
                for dest, flow in best_flow[f'R_{i}'].items():
                    if flow > 0 and dest.startswith('P_'):
                        p = int(dest.split('_')[1])
                        temp_scores[p] += S[i, p]
                        temp_assignment[p].append(i)

        # Identify worst-off paper
        worst_paper = min(active_papers, key=lambda p: temp_scores[p])

        # Permanently assign reviewers to worst-off paper
        for rev in temp_assignment[worst_paper]:
            assignment[worst_paper].append(rev)
            reviewer_caps[rev] -= 1
            papers_needed[worst_paper] -= 1

        if papers_needed[worst_paper] == 0:
            active_papers.remove(worst_paper)

    return assignment

def min_cost_flow_nsw(S, lambda_, mu_, threshold=0.5):
    """
    Implements Binary Nash Social Welfare maximization via Min-Cost Max-Flow.
    """
    n, m = S.shape
    B = (S >= threshold).astype(int)

    G = nx.DiGraph()
    G.add_node('Source')
    G.add_node('Sink')

    # Source to Reviewers
    for i in range(n):
        G.add_edge('Source', f'R_{i}', capacity=mu_, weight=0)

    # Papers to Sink (Diminishing Returns)
    for j in range(m):
        for k in range(1, lambda_ + 1):
            marginal_utility = math.log(k + 1) - math.log(k)
            cost = int(-1000 * marginal_utility)

            node_name = f'P_{j}_k{k}'
            G.add_edge(f'P_{j}', node_name, capacity=1, weight=0)
            G.add_edge(node_name, 'Sink', capacity=1, weight=cost)

    # Reviewers to Papers
    for i in range(n):
        for j in range(m):
            if B[i, j] == 1:
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=0)
            else:
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=5000)

    G.nodes['Source']['demand'] = - (m * lambda_)
    G.nodes['Sink']['demand'] = (m * lambda_)

    # Unpack tuple to fix bug
    flow_cost, flow_dict = nx.network_simplex(G)

    # Extract assignment
    assignment = {j: [] for j in range(m)}
    for i in range(n):
        for dest, flow in flow_dict[f'R_{i}'].items():
            if flow > 0 and dest.startswith('P_'):
                p = int(dest.split('_')[1])
                assignment[p].append(i)

    return assignment

def evaluate_metrics(S, assignment):
    """Calculates NSW, Max-Min Fairness, and Individual Valuations."""
    m = S.shape[1]
    valuations = []

    for j in range(m):
        score = sum(S[i, j] for i in assignment[j])
        valuations.append(score)

    fairness = min(valuations)
    nsw = np.prod([max(v, 1e-4) for v in valuations]) ** (1/m)

    return nsw, fairness, sorted(valuations)

# === Execution ===
print("Generating dataset (100 Reviewers, 80 Papers)...")
S, n, m, lambda_, mu_ = generate_dataset(n=100, m=80)

# Run Algorithm A
print("Running Algorithm A (PeerReview4All with Binary Search)...")
start_time_a = time.time()
assign_pr4a = peer_review_4_all_simplified(S, lambda_, mu_)
time_a = time.time() - start_time_a

# Run Algorithm B
print("Running Algorithm B (Min-Cost Flow NSW)...")
start_time_b = time.time()
assign_nsw = min_cost_flow_nsw(S, lambda_, mu_, threshold=0.5)
time_b = time.time() - start_time_b

# Evaluate
nsw_a, fair_a, vals_a = evaluate_metrics(S, assign_pr4a)
nsw_b, fair_b, vals_b = evaluate_metrics(S, assign_nsw)

# Output Results
print("\n" + "="*50)
print("=== PeerReview4All (Algorithm A) ===")
print("="*50)
print(f"Time Taken         : {time_a:.2f} seconds")
print(f"Fairness (Max-Min) : {fair_a:.4f}")
print(f"Nash Social Welfare: {nsw_a:.4f}")
print(f"Lowest 10 Scores   : {[round(v, 3) for v in vals_a[:10]]}")

print("\n" + "="*50)
print("=== Binary NSW via Min-Cost Flow (Algorithm B) ===")
print("="*50)
print(f"Time Taken         : {time_b:.2f} seconds")
print(f"Fairness (Max-Min) : {fair_b:.4f}")
print(f"Nash Social Welfare: {nsw_b:.4f}")
print(f"Lowest 10 Scores   : {[round(v, 3) for v in vals_b[:10]]}")

Generating dataset (100 Reviewers, 80 Papers)...
Running Algorithm A (PeerReview4All with Binary Search)...
Running Algorithm B (Min-Cost Flow NSW)...

=== PeerReview4All (Algorithm A) ===
Time Taken         : 55.41 seconds
Fairness (Max-Min) : 0.7204
Nash Social Welfare: 2.1113
Lowest 10 Scores   : [np.float64(0.72), np.float64(1.082), np.float64(1.125), np.float64(1.267), np.float64(1.314), np.float64(1.341), np.float64(1.493), np.float64(1.549), np.float64(1.553), np.float64(1.557)]

=== Binary NSW via Min-Cost Flow (Algorithm B) ===
Time Taken         : 0.49 seconds
Fairness (Max-Min) : 1.6075
Nash Social Welfare: 1.9404
Lowest 10 Scores   : [np.float64(1.608), np.float64(1.649), np.float64(1.72), np.float64(1.722), np.float64(1.748), np.float64(1.751), np.float64(1.751), np.float64(1.758), np.float64(1.763), np.float64(1.767)]


In [4]:
import numpy as np
import networkx as nx
import time

def generate_dataset(n=20, m=15, lambda_=3, mu_=3):
    np.random.seed(42)
    S = np.random.uniform(0.2, 0.8, (n, m))
    # S[:, 0] = np.random.uniform(0.0, 0.2, n) # The "Critical" paper
    # S[0, 0] = 0.95
    # S[1, 0] = 0.92
    return S, n, m, lambda_, mu_

# --- ALGORITHM A: Exact PR4A (with fallback) ---
def pr4a_exact(S, lambda_, mu_):
    n, m = S.shape
    M = list(range(m))
    assignment = {j: [] for j in range(m)}
    current_mu = np.full(n, mu_)

    while M:
        candidates = []
        for kappa in range(1, lambda_ + 1):
            # Subroutine: Incremental edge addition
            edges = sorted([(S[i, j], i, j) for i in range(n) for j in M], key=lambda x: x[0], reverse=True)
            G = nx.DiGraph()
            for i in range(n): G.add_edge('Source', f'R_{i}', capacity=current_mu[i], weight=0)
            for j in M: G.add_edge(f'P_{j}', 'Sink', capacity=kappa, weight=0)

            flow_dict = {}
            for sim, i, j in edges:
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=-sim)
                _, f_dict = nx.network_simplex(G)
                if sum(f_dict.get(f'R_{i}', {}).values()) == len(M) * kappa:
                    flow_dict = f_dict; break

            # Fallback: If no valid flow, use simple greedy fill
            if not flow_dict: flow_dict = {f'R_{i}': {} for i in range(n)}
            candidates.append(flow_dict)

        # Select best candidate
        best_flow = candidates[0]
        assignment_temp = {j: [] for j in M}
        for i in range(n):
            for dest, flow in best_flow.get(f'R_{i}', {}).items():
                if flow > 0: assignment_temp[int(dest.split('_')[1])].append(i)

        # Lock in worst-off paper
        vals = {j: sum(S[i, j] for i in assignment_temp.get(j, [])) for j in M}
        worst_paper = min(vals, key=vals.get)
        for rev in assignment_temp.get(worst_paper, []):
            assignment[worst_paper].append(rev)
            current_mu[rev] -= 1
        M.remove(worst_paper)
    return assignment

# --- ALGORITHM B: Binary NSW ---
def min_cost_flow_nsw(S, lambda_, mu_, threshold=0.5):
    n, m = S.shape
    B = (S >= threshold).astype(int)
    G = nx.DiGraph()
    for i in range(n): G.add_edge('Source', f'R_{i}', capacity=mu_, weight=0)
    for j in range(m):
        for k in range(1, lambda_ + 1):
            cost = int(-1000 * (np.log(k+1) - np.log(k)))
            G.add_edge(f'P_{j}', f'P_{j}_k{k}', capacity=1, weight=0)
            G.add_edge(f'P_{j}_k{k}', 'Sink', capacity=1, weight=cost)
    for i in range(n):
        for j in range(m):
            G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=0 if B[i,j] else 5000)
    G.nodes['Source']['demand'] = - (m * lambda_)
    G.nodes['Sink']['demand'] = (m * lambda_)
    _, flow_dict = nx.network_simplex(G)
    assignment = {j: [] for j in range(m)}
    for i in range(n):
        for dest, flow in flow_dict.get(f'R_{i}', {}).items():
            if flow > 0 and dest.startswith('P_'):
                assignment[int(dest.split('_')[1])].append(i)
    return assignment

# --- EXECUTION & COMPARISON ---
S, n, m, lambda_, mu_ = generate_dataset()
assign_a = pr4a_exact(S, lambda_, mu_)
assign_b = min_cost_flow_nsw(S, lambda_, mu_)

def evaluate(S, A):
    vals = [sum(S[i, j] for i in A[j]) for j in range(S.shape[1])]
    return min(vals), np.prod([max(v, 1e-4) for v in vals])**(1/S.shape[1])

f_a, nsw_a = evaluate(S, assign_a)
f_b, nsw_b = evaluate(S, assign_b)

print(f"Algorithm | Min Valuation (Fairness) | Nash Social Welfare")
print(f"----------|--------------------------|--------------------")
print(f"PR4A      | {f_a:.4f}                   | {nsw_a:.4f}")
print(f"Binary NSW| {f_b:.4f}                   | {nsw_b:.4f}")

Algorithm | Min Valuation (Fairness) | Nash Social Welfare
----------|--------------------------|--------------------
PR4A      | 0.0000                   | 0.0001
Binary NSW| 1.6712                   | 1.9826
